# 📖 Novel-TUI — Servidor LLM Remoto (GPU T4 en Google Colab)

Este cuaderno ejecuta **KoboldCpp** con aceleración CUDA en GPU T4 y el modelo **Llama-3-8B-Stheno-v3.2** (GGUF Q5_K_M sin censura).

### ⚡ Ventajas:
1. **Persistencia en Google Drive:** El modelo se descarga una sola vez en tu Drive (`/NovelTUI_Models/`) y en los próximos arranques inicia en 5 segundos.
2. **Túnel Cloudflare Gratuito:** Genera una URL pública `https://*.trycloudflare.com/v1` compatible con la API de OpenAI para Novel-TUI.

In [ ]:
    #@title 🚀 Iniciar Servidor KoboldCpp con GPU y Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR = '/content/drive/MyDrive/NovelTUI_Models'
MODEL_PATH = os.path.join(DRIVE_DIR, 'L3-8B-Stheno-v3.2-Q5_K_M.gguf')
MODEL_URL = 'https://huggingface.co/bartowski/L3-8B-Stheno-v3.2-GGUF/resolve/main/L3-8B-Stheno-v3.2-Q5_K_M.gguf'
KOBOLD_URL = 'https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64'

!mkdir -p /content/novel-llm
!mkdir -p "{DRIVE_DIR}"
%cd /content/novel-llm

   # 1. Si existe un archivo roto de menos de 1 GB en Drive, eliminarlo
if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) < 1000000000:
       print('⚠️ Detectado archivo corrupto/incompleto previo en Drive. Eliminando...')
       os.remove(MODEL_PATH)

   # 2. Descargar KoboldCpp
if not os.path.exists('koboldcpp_linux') or os.path.getsize('koboldcpp_linux') < 1000000:
       print('📥 Descargando KoboldCpp...')
       !wget -q -c {KOBOLD_URL} -O koboldcpp_linux
       !chmod +x koboldcpp_linux

   # 3. Descargar el modelo real de 5.7 GB a Google Drive si no existe
if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 1000000000:
       print('⚡ Modelo completo encontrado en Google Drive! Enlazando...')
       !ln -sf "{MODEL_PATH}" model.gguf
else:
       print('📥 Descargando L3-8B-Stheno a Google Drive (5.7 GB - tarda ~1 min en Colab)...')
       !wget -c "{MODEL_URL}" -O "{MODEL_PATH}"
       !ln -sf "{MODEL_PATH}" model.gguf

   # 4. Iniciar KoboldCpp con streaming de logs en vivo
print('🚀 Iniciando servidor KoboldCpp con GPU y Túnel Cloudflare...')
!./koboldcpp_linux --model model.gguf --usecuda 0 mmq --gpulayers 999 --contextsize 8192 --remotetunnel

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/novel-llm
⚡ Modelo completo encontrado en Google Drive! Enlazando...
🚀 Iniciando servidor KoboldCpp con GPU y Túnel Cloudflare...
***
Welcome to KoboldCpp - Version 1.120
cloudflared-linux-amd64 already exists, using existing file.
Attempting to start tunnel thread...
Loading Chat Completions Adapter: /tmp/_MEIWgQmt5/kcpp_adapters/AutoGuess.json
Chat Completions Adapter Loaded
System: Linux #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64
Detected Available GPU Memory: 15360 MB
Detected Available RAM: 11618 MB
Initializing dynamic library: koboldcpp_cublas.so
Starting Cloudflare Tunnel for Linux, please wait...
Namespace(admin=False, admindir='', adminpassword=None, adminunloadtimeout=0, allow_config_onready=False, analyze='', autofit=False, autofitpadding=1024, autoswapmode=False, baseconfig='', batchsize=512, benchmark=None, blasthreads=0, chatcom